In [1]:
import sys
import os

# Aggiungi project root al path per gli import
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project Root aggiunta al path: {project_root}")

Project Root aggiunta al path: c:\Users\emagi\Documents\Deep_Learning\Progetto_deep_learning


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from src.ModelClasses.naive import NaivePersistence
from src.Training.engine import validate_one_epoch
from src.DataLoading.data_loader import TS_Cross_Validator
from src.config import TARGET_COL, SAMPLING_CONFIG

# 1. Caricamento Dati 
df = pd.read_csv("../data/processed/preprocessed_ds.csv")
print(f"Dataset shape: {df.shape}")
print(f"Target column: {TARGET_COL}")

# 2. Creazione Folds
validator = TS_Cross_Validator(df, TARGET_COL, SAMPLING_CONFIG)
folds = list(validator.get_folds())  # Converti generator in lista

# 3. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Usiamo L1Loss (MAE) per il denominatore del MASE
mae_metric = nn.L1Loss()

naive_maes = []

print(f"\n--- Calcolo Benchmark Naive su {len(folds)} Fold ---\n")

for i, (train_loader, val_loader, scaler) in enumerate(folds):
    # Crea modello Naive con loader per rilevare target_idx corretto
    naive_model = NaivePersistence(train_loader=train_loader).to(device)
    
    # validate_one_epoch restituisce 3 valori: (avg_loss, avg_mae, avg_rmse)
    _, fold_mae, _ = validate_one_epoch(naive_model, val_loader, nn.MSELoss(), device)
    
    naive_maes.append(fold_mae)
    print(f"FOLD {i+1} -> Naive MAE: {fold_mae:.6f}")

print("\n--- COPIA QUESTO NEL TUO training_config.py ---")
print(f"NAIVE_MAE_PER_FOLD = {naive_maes}")

c:\Users\emagi\Documents\Deep_Learning\Progetto_deep_learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset shape: (17544, 25)
Target column: pv_power

=== INIZIO CROSS-VALIDATION (3 splits) ===

---------------- FOLD 1 ----------------
TRAIN: 0 -> 4385
VAL  : 4338 -> 8771
Fold 1 pronto. Yielding...

---------------- FOLD 2 ----------------
TRAIN: 0 -> 8771
VAL  : 8724 -> 13157
Fold 2 pronto. Yielding...

---------------- FOLD 3 ----------------
TRAIN: 0 -> 13157
VAL  : 13110 -> 17543
Fold 3 pronto. Yielding...
Device: cuda

--- Calcolo Benchmark Naive su 3 Fold ---

NaivePersistence - target_idx rilevato dal loader: 24
FOLD 1 -> Naive MAE: 0.061714
NaivePersistence - target_idx rilevato dal loader: 24
FOLD 2 -> Naive MAE: 0.073218
NaivePersistence - target_idx rilevato dal loader: 24
FOLD 3 -> Naive MAE: 0.067258

--- COPIA QUESTO NEL TUO training_config.py ---
NAIVE_MAE_PER_FOLD = [0.06171385527493945, 0.07321843994187488, 0.06725753733105418]
